# CineSense-AI: Context-Aware Recommendation System

## Objective

This notebook extends the core recommendation engine by incorporating the user's current context and preferences.

The system will consider factors such as:

- Mood
- Available watching time
- Watching companion
- Desired movie experience
- Preferred and avoided genres
- Language preference

The goal is to recommend movies that are suitable not only for the user's historical taste, but also for their current situation.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving ratings_clean.csv to ratings_clean.csv


In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv("movies_enriched.csv")
ratings = pd.read_csv("ratings_clean.csv")
movie_genres = pd.read_csv("movie_genres.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Movie Genres:", movie_genres.shape)

display(movies.head())

Movies: (9742, 25)
Ratings: (100836, 5)
Movie Genres: (22084, 2)


,movieId,title,genres,year,clean_title,genre_list,combined_tags,rating_count,average_rating,tmdbId,...,tmdb_popularity,vote_average,vote_count,original_language,poster_path,backdrop_path,adult,status,tagline,tmdb_genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",pixar pixar fun,215,3.92,862.0,...,30.9498,7.983,20174.0,en,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,False,Released,The adventure takes off when toys come to life!,Family|Comedy|Animation|Adventure
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",fantasy magic board game robin williams game,110,3.43,8844.0,...,2.4359,7.249,11395.0,en,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,False,Released,It's a jungle in here.,Adventure|Fantasy|Family
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",moldy old,52,3.26,15602.0,...,1.6558,6.479,432.0,en,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1o4vuCHpmd4DXofMYDUwpnhKiuy.jpg,False,Released,Still Yelling. Still Fighting. Still Ready for...,Romance|Comedy
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",NaN,7,2.36,31357.0,...,1.8246,6.261,207.0,en,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg,/jZjoEKXMTDoZAGdkjhAdJaKtXSN.jpg,False,Released,Friends are the people who let you be yourself...,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],pregnancy remake,49,3.07,11862.0,...,2.1537,6.271,819.0,en,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,/lEsjVrGU21BeJjF5AF9EWsihDpw.jpg,False,Released,Just when his world is back to normal... he's ...,Comedy|Family


In [ ]:
context_columns = [
    "clean_title",
    "genres",
    "runtime",
    "overview",
    "tagline",
    "original_language",
    "tmdb_popularity",
    "vote_average",
    "vote_count",
    "average_rating",
    "rating_count"
]

display(movies[context_columns].head(10))

,clean_title,genres,runtime,overview,tagline,original_language,tmdb_popularity,vote_average,vote_count,average_rating,rating_count
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,en,30.9498,7.983,20174.0,3.92,215
1,Jumanji,Adventure|Children|Fantasy,104.0,When siblings Judy and Peter discover an encha...,It's a jungle in here.,en,2.4359,7.249,11395.0,3.43,110
2,Grumpier Old Men,Comedy|Romance,101.0,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,en,1.6558,6.479,432.0,3.26,52
3,Waiting to Exhale,Comedy|Drama|Romance,127.0,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,en,1.8246,6.261,207.0,2.36,7
4,Father of the Bride Part II,Comedy,106.0,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,en,2.1537,6.271,819.0,3.07,49
5,Heat,Action|Crime|Thriller,170.0,Obsessive master thief Neil McCauley leads a t...,A Los Angeles crime saga.,en,11.9265,7.900,8470.0,3.95,102
6,Sabrina,Comedy|Romance,127.0,"After her return from school in Paris, a playb...",You are cordially invited to the most surprisi...,en,2.6983,6.212,702.0,3.19,54
7,Tom and Huck,Adventure|Children,97.0,"A mischievous young boy, Tom Sawyer, witnesses...",A lot of kids get into trouble. These two inve...,en,0.9201,5.302,215.0,2.88,8
8,Sudden Death,Action,111.0,When a man's daughter is suddenly taken during...,Terror goes into overtime.,en,1.6499,6.028,816.0,3.12,16
9,GoldenEye,Action|Adventure|Thriller,130.0,When a powerful secret defense system is stole...,No limits. No fears. No substitutes.,en,5.0216,6.905,4386.0,3.50,132


In [ ]:
print("Total Movies:", len(movies))

print("\nAvailable Context Data:")
print(
    movies[
        [
            "runtime",
            "overview",
            "tagline",
            "original_language",
            "tmdb_popularity"
        ]
    ]
    .notna()
    .sum()
)

print("\nUnique Languages:")
print(movies["original_language"].nunique())

print("\nMost Common Languages:")
print(
    movies["original_language"]
    .value_counts()
    .head(10)
)

Total Movies: 9742

Available Context Data:
runtime              9615
overview             9614
tagline              8365
original_language    9615
tmdb_popularity      9615
dtype: int64

Unique Languages:
48

Most Common Languages:
original_language
en    8183
fr     337
ja     260
it     142
ru     112
de     109
es      86
cn      66
zh      65
ko      38
Name: count, dtype: int64


## 5. Context Preference Schema

The recommendation system represents the user's current movie-watching context using structured variables.

The context includes:

- Current mood
- Watching companion
- Available watching time
- Desired experience
- Preferred genres
- Genres to avoid
- Preferred language

These contextual signals will later be converted into recommendation scores and combined with movie quality and personalization signals.

In [ ]:
MOOD_OPTIONS = [
    "happy",
    "sad",
    "stressed",
    "bored",
    "excited",
    "romantic",
    "thoughtful",
    "scared",
    "relaxed"
]

COMPANION_OPTIONS = [
    "alone",
    "friends",
    "family",
    "partner"
]

EXPERIENCE_OPTIONS = [
    "funny",
    "exciting",
    "relaxing",
    "emotional",
    "romantic",
    "thought_provoking",
    "suspenseful",
    "inspiring"
]

print("Mood Options:")
print(MOOD_OPTIONS)

print("\nCompanion Options:")
print(COMPANION_OPTIONS)

print("\nExperience Options:")
print(EXPERIENCE_OPTIONS)

Mood Options:
['happy', 'sad', 'stressed', 'bored', 'excited', 'romantic', 'thoughtful', 'scared', 'relaxed']

Companion Options:
['alone', 'friends', 'family', 'partner']

Experience Options:
['funny', 'exciting', 'relaxing', 'emotional', 'romantic', 'thought_provoking', 'suspenseful', 'inspiring']


In [ ]:
print(movie_genres.columns.tolist())

['movieId', 'genre']


In [ ]:
available_genres = sorted(
    movie_genres["genre"]
    .dropna()
    .unique()
    .tolist()
)

print("Number of Genres:", len(available_genres))
print("\nAvailable Genres:")
print(available_genres)

Number of Genres: 20

Available Genres:
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [ ]:
available_genres = sorted(
    genre
    for genre in movie_genres["genre"].dropna().unique()
    if genre != "(no genres listed)"
)

print("Number of Valid Genres:", len(available_genres))
print("\nAvailable Genres:")
print(available_genres)

Number of Valid Genres: 19

Available Genres:
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [ ]:
available_languages = sorted(
    movies["original_language"]
    .dropna()
    .unique()
    .tolist()
)

print("Number of Languages:", len(available_languages))
print(available_languages)

Number of Languages: 48
['ar', 'bg', 'bn', 'bo', 'bs', 'ca', 'cn', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fa', 'fi', 'fr', 'he', 'hi', 'hu', 'hy', 'id', 'is', 'it', 'iu', 'ja', 'ko', 'ku', 'mk', 'mn', 'nl', 'no', 'pl', 'ps', 'pt', 'ro', 'ru', 'sh', 'sr', 'sv', 'ta', 'th', 'tn', 'tr', 'vi', 'wo', 'xx', 'zh']


In [ ]:
user_context = {
    "mood": "stressed",
    "companion": "friends",
    "max_runtime": 120,
    "experience": "funny",

    "preferred_genres": [
        "Comedy",
        "Adventure"
    ],

    "avoid_genres": [
        "Horror"
    ],

    "preferred_language": "en"
}

user_context

{'mood': 'stressed',
 'companion': 'friends',
 'max_runtime': 120,
 'experience': 'funny',
 'preferred_genres': ['Comedy', 'Adventure'],
 'avoid_genres': ['Horror'],
 'preferred_language': 'en'}

In [ ]:
def validate_context(context):
    errors = []

    if context["mood"] not in MOOD_OPTIONS:
        errors.append("Invalid mood.")

    if context["companion"] not in COMPANION_OPTIONS:
        errors.append("Invalid companion.")

    if context["experience"] not in EXPERIENCE_OPTIONS:
        errors.append("Invalid experience.")

    if not isinstance(context["max_runtime"], (int, float)):
        errors.append("Runtime must be numeric.")

    elif context["max_runtime"] <= 0:
        errors.append("Runtime must be greater than 0.")

    invalid_preferred = [
        genre
        for genre in context["preferred_genres"]
        if genre not in available_genres
    ]

    invalid_avoided = [
        genre
        for genre in context["avoid_genres"]
        if genre not in available_genres
    ]

    if invalid_preferred:
        errors.append(
            f"Invalid preferred genres: {invalid_preferred}"
        )

    if invalid_avoided:
        errors.append(
            f"Invalid avoided genres: {invalid_avoided}"
        )

    if context["preferred_language"] not in available_languages:
        errors.append("Invalid language.")

    return errors

In [ ]:
validation_errors = validate_context(user_context)

if len(validation_errors) == 0:
    print("✓ User context is valid.")
else:
    print("Context validation failed:")

    for error in validation_errors:
        print("-", error)

✓ User context is valid.


## Step 6: Context-Aware Movie Filtering

This step filters movies according to the user's current context, including available time, language, preferred genres, avoided genres, and minimum rating requirements.

In [ ]:
def filter_movies_by_context(
    movies_df,
    available_time=None,
    language=None,
    preferred_genres=None,
    avoided_genres=None,
    min_rating=0
):
    filtered = movies_df.copy()

    # 1. Runtime filtering
    if available_time is not None:
        filtered = filtered[
            filtered["runtime"].notna() &
            (filtered["runtime"] <= available_time)
        ]

    # 2. Language filtering
    if language is not None:
        filtered = filtered[
            filtered["original_language"] == language
        ]

    # 3. Preferred genre filtering
    if preferred_genres:
        pattern = "|".join(preferred_genres)

        filtered = filtered[
            filtered["genres"]
            .fillna("")
            .str.contains(pattern, case=False, regex=True)
        ]

    # 4. Avoided genre filtering
    if avoided_genres:
        avoid_pattern = "|".join(avoided_genres)

        filtered = filtered[
            ~filtered["genres"]
            .fillna("")
            .str.contains(avoid_pattern, case=False, regex=True)
        ]

    # 5. Minimum rating
    if min_rating is not None:
        filtered = filtered[
            filtered["vote_average"].fillna(0) >= min_rating
        ]

    return filtered.copy()

In [ ]:
test_context = {
    "available_time": 120,
    "language": "en",
    "preferred_genres": ["Comedy", "Adventure"],
    "avoided_genres": ["Horror"],
    "min_rating": 6.0
}

context_filtered = filter_movies_by_context(
    movies,
    **test_context
)

print("Movies before filtering:", len(movies))
print("Movies after context filtering:", len(context_filtered))

display(
    context_filtered[
        [
            "clean_title",
            "genres",
            "runtime",
            "original_language",
            "vote_average",
            "tmdb_popularity"
        ]
    ].head(10)
)

Movies before filtering: 9742
Movies after context filtering: 2207


,clean_title,genres,runtime,original_language,vote_average,tmdb_popularity
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,en,7.983,30.9498
1,Jumanji,Adventure|Children|Fantasy,104.0,en,7.249,2.4359
2,Grumpier Old Men,Comedy|Romance,101.0,en,6.479,1.6558
4,Father of the Bride Part II,Comedy,106.0,en,6.271,2.1537
10,"American President, The",Comedy|Drama|Romance,113.0,en,6.540,2.1031
12,Balto,Adventure|Animation|Children,78.0,en,7.333,2.5808
18,Ace Ventura: When Nature Calls,Comedy,90.0,en,6.343,3.9605
20,Get Shorty,Comedy|Crime|Thriller,105.0,en,6.500,1.6253
34,It Takes Two,Children|Comedy,101.0,en,6.504,2.5553
35,Clueless,Comedy|Romance,97.0,en,7.250,5.0402


### 6.3 Mood and Experience-Aware Scoring

After context filtering, the remaining movies are ranked according to the user's current mood and desired viewing experience.

Mood and experience preferences are mapped to relevant movie genres and textual characteristics.

In [ ]:
mood_genre_map = {
    "happy": ["Comedy", "Adventure", "Animation", "Musical"],
    "sad": ["Comedy", "Animation", "Family", "Romance"],
    "stressed": ["Comedy", "Animation", "Family", "Fantasy"],
    "excited": ["Action", "Adventure", "Thriller", "Sci-Fi"],
    "romantic": ["Romance", "Comedy", "Drama"],
    "bored": ["Action", "Adventure", "Mystery", "Thriller"],
    "thoughtful": ["Drama", "Mystery", "Documentary", "Sci-Fi"]
}

experience_genre_map = {
    "funny": ["Comedy"],
    "exciting": ["Action", "Adventure", "Thriller"],
    "relaxing": ["Comedy", "Animation", "Family", "Romance"],
    "emotional": ["Drama", "Romance"],
    "romantic": ["Romance"],
    "thought-provoking": ["Drama", "Mystery", "Documentary", "Sci-Fi"],
    "suspenseful": ["Thriller", "Mystery", "Crime", "Horror"],
    "inspiring": ["Drama", "Adventure", "Biography", "Documentary"]
}

print("Mood mappings:", len(mood_genre_map))
print("Experience mappings:", len(experience_genre_map)
)

Mood mappings: 7
Experience mappings: 8


In [ ]:
def calculate_context_score(
    df,
    mood=None,
    experience=None
):
    scored = df.copy()

    scored["context_score"] = 0.0

    # Mood score
    if mood in mood_genre_map:
        mood_genres = mood_genre_map[mood]

        scored["context_score"] += scored["genres"].fillna("").apply(
            lambda x: sum(
                genre.lower() in x.lower()
                for genre in mood_genres
            )
        ) * 2

    # Experience score
    if experience in experience_genre_map:
        experience_genres = experience_genre_map[experience]

        scored["context_score"] += scored["genres"].fillna("").apply(
            lambda x: sum(
                genre.lower() in x.lower()
                for genre in experience_genres
            )
        ) * 2

    return scored

In [ ]:
scored_movies = calculate_context_score(
    context_filtered,
    mood="happy",
    experience="funny"
)

scored_movies = scored_movies.sort_values(
    ["context_score", "vote_average", "tmdb_popularity"],
    ascending=[False, False, False]
)

display(
    scored_movies[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "tmdb_popularity",
            "context_score"
        ]
    ].head(10)
)

,clean_title,genres,runtime,vote_average,tmdb_popularity,context_score
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,13.0892,10.0
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,11.2691,10.0
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,8.6323,10.0
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,12.7737,10.0
2287,Robin Hood,Adventure|Animation|Children|Comedy|Musical,83.0,7.300,4.0985,10.0
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,19.7226,10.0
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,1.8684,10.0
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,4.9232,10.0
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,3.6961,10.0
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,7.983,30.9498,8.0


In [ ]:
def calculate_final_score(df):
    ranked = df.copy()

    # Normalize TMDB rating (0–10 → 0–1)
    ranked["rating_score"] = (
        ranked["vote_average"].fillna(0) / 10
    )

    # Normalize popularity
    max_popularity = ranked["tmdb_popularity"].max()

    if max_popularity > 0:
        ranked["popularity_score"] = (
            ranked["tmdb_popularity"].fillna(0) / max_popularity
        )
    else:
        ranked["popularity_score"] = 0

    # Normalize context score
    max_context = ranked["context_score"].max()

    if max_context > 0:
        ranked["normalized_context_score"] = (
            ranked["context_score"] / max_context
        )
    else:
        ranked["normalized_context_score"] = 0

    # Final weighted score
    ranked["final_score"] = (
        0.60 * ranked["normalized_context_score"] +
        0.30 * ranked["rating_score"] +
        0.10 * ranked["popularity_score"]
    )

    return ranked

In [ ]:
final_recommendations = calculate_final_score(scored_movies)

final_recommendations = final_recommendations.sort_values(
    "final_score",
    ascending=False
)

display(
    final_recommendations[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "tmdb_popularity",
            "context_score",
            "final_score"
        ]
    ].head(10)
)

,clean_title,genres,runtime,vote_average,tmdb_popularity,context_score,final_score
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,19.7226,10.0,0.881194
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,13.0892,10.0,0.879442
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,11.2691,10.0,0.866061
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,12.7737,10.0,0.861052
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,8.6323,10.0,0.852771
2287,Robin Hood,Adventure|Animation|Children|Comedy|Musical,83.0,7.300,4.0985,10.0,0.832242
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,1.8684,10.0,0.822637
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,4.9232,10.0,0.821287
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,7.983,30.9498,8.0,0.819490
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,3.6961,10.0,0.814052


## Step 7: Complete Context-Aware Recommendation Engine

This step combines context filtering, mood and experience scoring, and final ranking into a single recommendation pipeline.

In [ ]:
def recommend_movies(
    movies_df,
    available_time=None,
    language=None,
    preferred_genres=None,
    avoided_genres=None,
    min_rating=0,
    mood=None,
    experience=None,
    top_n=10
):

    # Step 1: Context filtering
    filtered = filter_movies_by_context(
        movies_df,
        available_time=available_time,
        language=language,
        preferred_genres=preferred_genres,
        avoided_genres=avoided_genres,
        min_rating=min_rating
    )

    if filtered.empty:
        print("No movies found matching this context.")
        return filtered

    # Step 2: Mood + experience scoring
    scored = calculate_context_score(
        filtered,
        mood=mood,
        experience=experience
    )

    # Step 3: Final ranking
    ranked = calculate_final_score(scored)

    ranked = ranked.sort_values(
        "final_score",
        ascending=False
    )

    return ranked.head(top_n)

In [ ]:
recommendations = recommend_movies(
    movies,
    available_time=120,
    language="en",
    preferred_genres=["Comedy", "Adventure"],
    avoided_genres=["Horror"],
    min_rating=6.0,
    mood="happy",
    experience="funny",
    top_n=10
)

display(
    recommendations[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "tmdb_popularity",
            "context_score",
            "final_score"
        ]
    ]
)

,clean_title,genres,runtime,vote_average,tmdb_popularity,context_score,final_score
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,19.7226,10.0,0.881194
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,13.0892,10.0,0.879442
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,11.2691,10.0,0.866061
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,12.7737,10.0,0.861052
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,8.6323,10.0,0.852771
2287,Robin Hood,Adventure|Animation|Children|Comedy|Musical,83.0,7.300,4.0985,10.0,0.832242
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,1.8684,10.0,0.822637
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,4.9232,10.0,0.821287
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,7.983,30.9498,8.0,0.819490
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,3.6961,10.0,0.814052


In [ ]:
# Choose one MovieLens user for testing
test_user_id = 1

user_ratings = ratings[
    ratings["userId"] == test_user_id
].copy()

print("User ID:", test_user_id)
print("Number of movies rated:", len(user_ratings))

display(
    user_ratings
    .sort_values("rating", ascending=False)
    .head(10)
)

User ID: 1
Number of movies rated: 232


,userId,movieId,rating,timestamp,datetime
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51
6,1,101,5.0,964980868,2000-07-30 18:14:28
13,1,231,5.0,964981179,2000-07-30 18:19:39
11,1,216,5.0,964981208,2000-07-30 18:20:08
10,1,163,5.0,964983650,2000-07-30 19:00:50
9,1,157,5.0,964984100,2000-07-30 19:08:20
8,1,151,5.0,964984041,2000-07-30 19:07:21
35,1,596,5.0,964982838,2000-07-30 18:47:18
31,1,553,5.0,964984153,2000-07-30 19:09:13


In [ ]:
user_history = user_ratings.merge(
    movies[
        [
            "movieId",
            "clean_title",
            "genres"
        ]
    ],
    on="movieId",
    how="left"
)

display(
    user_history[
        [
            "movieId",
            "clean_title",
            "genres",
            "rating"
        ]
    ]
    .sort_values("rating", ascending=False)
    .head(15)
)

,movieId,clean_title,genres,rating
3,47,Seven (a.k.a. Se7en),Mystery|Thriller,5.0
4,50,"Usual Suspects, The",Crime|Mystery|Thriller,5.0
6,101,Bottle Rocket,Adventure|Comedy|Crime|Romance,5.0
13,231,Dumb & Dumber (Dumb and Dumber),Adventure|Comedy,5.0
11,216,Billy Madison,Comedy,5.0
10,163,Desperado,Action|Romance|Western,5.0
9,157,Canadian Bacon,Comedy|War,5.0
8,151,Rob Roy,Action|Drama|Romance|War,5.0
35,596,Pinocchio,Animation|Children|Fantasy|Musical,5.0
31,553,Tombstone,Action|Drama|Western,5.0


### 8.3 User Genre Taste Profile

The user's historical ratings are analyzed to estimate their preference for each movie genre. Higher average ratings indicate stronger genre preferences.

In [ ]:
# Split each movie into individual genres
genre_history = user_history.copy()

genre_history["genre"] = (
    genre_history["genres"]
    .fillna("")
    .str.split("|")
)

genre_history = genre_history.explode("genre")

# Remove empty genres
genre_history = genre_history[
    genre_history["genre"] != ""
]

# Calculate preference statistics
genre_profile = (
    genre_history
    .groupby("genre")
    .agg(
        average_user_rating=("rating", "mean"),
        movies_rated=("rating", "count")
    )
    .reset_index()
)

# Sort by rating, then amount of evidence
genre_profile = genre_profile.sort_values(
    ["average_user_rating", "movies_rated"],
    ascending=[False, False]
)

display(genre_profile)

,genre,average_user_rating,movies_rated
8,Film-Noir,5.000000,1
2,Animation,4.689655,29
10,Musical,4.681818,22
3,Children,4.547619,42
6,Drama,4.529412,68
15,War,4.500000,22
1,Adventure,4.388235,85
5,Crime,4.355556,45
0,Action,4.322222,90
12,Romance,4.307692,26


In [ ]:
# User's overall average rating
user_mean_rating = user_history["rating"].mean()

# Reliability-adjusted genre preference
genre_profile["taste_score"] = (
    genre_profile["average_user_rating"] - user_mean_rating
) * np.log1p(genre_profile["movies_rated"])

genre_profile = genre_profile.sort_values(
    "taste_score",
    ascending=False
)

print("User's overall average rating:", round(user_mean_rating, 2))

display(
    genre_profile[
        [
            "genre",
            "average_user_rating",
            "movies_rated",
            "taste_score"
        ]
    ]
)

User's overall average rating: 4.37


,genre,average_user_rating,movies_rated,taste_score
2,Animation,4.689655,29,1.099525
10,Musical,4.681818,22,0.989057
6,Drama,4.529412,68,0.690297
3,Children,4.547619,42,0.681679
8,Film-Noir,5.000000,1,0.439192
15,War,4.500000,22,0.418967
1,Adventure,4.388235,85,0.097354
5,Crime,4.355556,45,-0.041440
16,Western,4.285714,7,-0.167738
12,Romance,4.307692,26,-0.193423


### 8.5 Personalized Movie Scoring

Each candidate movie receives a personalization score based on the user's historical genre preferences.

In [ ]:
# Create dictionary:
# genre -> user's taste score

genre_score_map = dict(
    zip(
        genre_profile["genre"],
        genre_profile["taste_score"]
    )
)

def calculate_personalization_score(genres):

    if pd.isna(genres):
        return 0.0

    movie_genres = str(genres).split("|")

    scores = [
        genre_score_map.get(g, 0)
        for g in movie_genres
    ]

    if len(scores) == 0:
        return 0.0

    return np.mean(scores)


# Test personalization on movies
personalized_movies = movies.copy()

personalized_movies["personalization_score"] = (
    personalized_movies["genres"]
    .apply(calculate_personalization_score)
)

display(
    personalized_movies[
        [
            "clean_title",
            "genres",
            "personalization_score"
        ]
    ]
    .sort_values(
        "personalization_score",
        ascending=False
    )
    .head(15)
)

,clean_title,genres,personalization_score
7195,Merry Madagascar,Animation,1.099525
9729,Bunny,Animation,1.099525
9624,"Fireworks, Should We See It from the Side or t...",Animation,1.099525
9601,LEGO DC Super Hero Girls: Brain Drain,Animation,1.099525
9549,Cheburashka,Animation,1.099525
8718,Hedgehog in the Fog,Animation,1.099525
8993,Ooops! Noah is Gone...,Animation,1.099525
9561,Travels of an Ant,Animation,1.099525
9343,The Red Turtle,Animation,1.099525
7439,"Illusionist, The (L'illusionniste)",Animation,1.099525


In [ ]:
# STEP 8.6
# Add personalization score to context-based recommendations

personalized_recommendations = recommendations.copy()

personalized_recommendations["personalization_score"] = (
    personalized_recommendations["genres"]
    .apply(calculate_personalization_score)
)

# Normalize personalization score to 0-1
min_p = personalized_recommendations["personalization_score"].min()
max_p = personalized_recommendations["personalization_score"].max()

if max_p != min_p:
    personalized_recommendations["personalization_normalized"] = (
        personalized_recommendations["personalization_score"] - min_p
    ) / (max_p - min_p)
else:
    personalized_recommendations["personalization_normalized"] = 0.5


# Combine existing final score + historical personalization
personalized_recommendations["hybrid_score"] = (
    0.70 * personalized_recommendations["final_score"] +
    0.30 * personalized_recommendations["personalization_normalized"]
)

# Sort by final hybrid score
personalized_recommendations = (
    personalized_recommendations
    .sort_values("hybrid_score", ascending=False)
)

display(
    personalized_recommendations[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "context_score",
            "personalization_score",
            "final_score",
            "hybrid_score"
        ]
    ].head(10)
)

,clean_title,genres,runtime,vote_average,context_score,personalization_score,final_score,hybrid_score
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,10.0,0.494414,0.866061,0.906243
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,10.0,0.494414,0.852771,0.896940
2287,Robin Hood,Adventure|Animation|Children|Comedy|Musical,83.0,7.300,10.0,0.494414,0.832242,0.882570
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,10.0,0.494414,0.814052,0.869837
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,10.0,0.424135,0.879442,0.838224
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,10.0,0.379775,0.861052,0.776506
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,10.0,0.305038,0.822637,0.667322
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,10.0,0.287635,0.821287,0.647215
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,10.0,0.221961,0.881194,0.616836
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,81.0,7.983,8.0,0.243562,0.819490,0.597428


In [ ]:
# STEP 8.7
# Remove movies already rated by the test user

watched_movie_ids = set(user_ratings["movieId"])

unseen_recommendations = personalized_recommendations[
    ~personalized_recommendations["movieId"].isin(watched_movie_ids)
].copy()

print("Recommendations before removing watched movies:",
      len(personalized_recommendations))

print("Recommendations after removing watched movies:",
      len(unseen_recommendations))

print("Already rated by User", test_user_id, ":",
      len(watched_movie_ids))


display(
    unseen_recommendations[
        [
            "movieId",
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "context_score",
            "personalization_score",
            "hybrid_score"
        ]
    ].head(10)
)

Recommendations before removing watched movies: 10
Recommendations after removing watched movies: 8
Already rated by User 1 : 232


,movieId,clean_title,genres,runtime,vote_average,context_score,personalization_score,hybrid_score
506,588,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,10.0,0.494414,0.906243
1177,1566,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,10.0,0.494414,0.896940
578,709,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,10.0,0.494414,0.869837
1390,1907,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,10.0,0.424135,0.838224
5160,8360,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,10.0,0.379775,0.776506
2144,2857,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,10.0,0.305038,0.667322
6626,56152,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,10.0,0.287635,0.647215
8303,106696,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,10.0,0.221961,0.616836


In [ ]:
def personalized_recommend_movies(
    user_id,
    available_time,
    language,
    preferred_genres,
    avoided_genres,
    min_rating,
    mood,
    experience,
    top_n=10
):

    # 1. Get context-based recommendations
    context_recs = recommend_movies(
        movies,
        available_time=available_time,
        language=language,
        preferred_genres=preferred_genres,
        avoided_genres=avoided_genres,
        min_rating=min_rating,
        mood=mood,
        experience=experience,
        top_n=50
    )

    # 2. Get user's rating history
    user_data = ratings[
        ratings["userId"] == user_id
    ].copy()

    # If new user has no history
    if len(user_data) == 0:
        return context_recs.head(top_n)

    # 3. Merge ratings with movie genres
    history = user_data.merge(
        movies[["movieId", "genres"]],
        on="movieId",
        how="left"
    )

    # 4. Build genre preference scores
    expanded = (
        history
        .assign(genre=history["genres"].str.split("|"))
        .explode("genre")
        .dropna(subset=["genre"])
    )

    user_mean = user_data["rating"].mean()

    profile = (
        expanded
        .groupby("genre")
        .agg(
            average_user_rating=("rating", "mean"),
            movies_rated=("rating", "count")
        )
        .reset_index()
    )

    profile["taste_score"] = (
        (profile["average_user_rating"] - user_mean)
        * np.log1p(profile["movies_rated"])
    )

    score_map = dict(
        zip(profile["genre"], profile["taste_score"])
    )

    # 5. Personalization score
    def movie_personalization(genres):

        if pd.isna(genres):
            return 0.0

        genres_list = str(genres).split("|")

        scores = [
            score_map.get(g, 0)
            for g in genres_list
        ]

        return np.mean(scores) if scores else 0.0

    context_recs = context_recs.copy()

    context_recs["personalization_score"] = (
        context_recs["genres"]
        .apply(movie_personalization)
    )

    # 6. Normalize personalization
    min_p = context_recs["personalization_score"].min()
    max_p = context_recs["personalization_score"].max()

    if max_p != min_p:
        context_recs["personalization_normalized"] = (
            context_recs["personalization_score"] - min_p
        ) / (max_p - min_p)
    else:
        context_recs["personalization_normalized"] = 0.5

    # 7. Hybrid score
    context_recs["hybrid_score"] = (
        0.70 * context_recs["final_score"] +
        0.30 * context_recs["personalization_normalized"]
    )

    # 8. Remove movies already watched/rated
    watched = set(user_data["movieId"])

    context_recs = context_recs[
        ~context_recs["movieId"].isin(watched)
    ]

    # 9. Final ranking
    return (
        context_recs
        .sort_values("hybrid_score", ascending=False)
        .head(top_n)
    )

In [ ]:
final_user_recs = personalized_recommend_movies(
    user_id=1,
    available_time=120,
    language="en",
    preferred_genres=["Comedy", "Adventure"],
    avoided_genres=["Horror"],
    min_rating=6.0,
    mood="happy",
    experience="funny",
    top_n=10
)

display(
    final_user_recs[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "context_score",
            "personalization_score",
            "final_score",
            "hybrid_score"
        ]
    ]
)

,clean_title,genres,runtime,vote_average,context_score,personalization_score,final_score,hybrid_score
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,91.0,7.655,10.0,0.494414,0.866061,0.847809
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,93.0,7.496,10.0,0.494414,0.852771,0.838507
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,88.0,7.905,10.0,0.424135,0.879442,0.815805
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,74.0,6.737,10.0,0.494414,0.814052,0.811403
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,92.0,7.326,10.0,0.379775,0.861052,0.776820
1545,"Little Mermaid, The",Animation|Children|Comedy|Musical|Romance,83.0,7.352,8.0,0.436259,0.730650,0.718788
4360,Finding Nemo,Adventure|Animation|Children|Comedy,100.0,7.820,8.0,0.370754,0.771596,0.708890
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,90.0,7.220,10.0,0.305038,0.822637,0.705934
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,102.0,7.249,10.0,0.221961,0.881194,0.698021
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,107.0,6.846,10.0,0.287635,0.821287,0.694745


In [ ]:
user_id=1
available_time=120
language="en"
preferred_genres=["Comedy", "Adventure"]
avoided_genres=["Horror"]
mood="happy"
experience="funny"

## Step 9 — Interactive CineSense Recommendation Interface

This section converts the recommendation engine into an interactive system where user preferences can be entered without modifying the recommendation algorithm.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

user_id_widget = widgets.IntText(
    value=1,
    description="User ID:"
)

time_widget = widgets.IntSlider(
    value=120,
    min=30,
    max=240,
    step=15,
    description="Time:"
)

language_widget = widgets.Dropdown(
    options=[
        ("English", "en"),
        ("French", "fr"),
        ("Japanese", "ja"),
        ("Italian", "it"),
        ("Spanish", "es"),
        ("German", "de"),
        ("Chinese", "zh"),
        ("Korean", "ko")
    ],
    value="en",
    description="Language:"
)

mood_widget = widgets.Dropdown(
    options=[
        "happy",
        "sad",
        "relaxed",
        "excited",
        "romantic",
        "thoughtful"
    ],
    value="happy",
    description="Mood:"
)

experience_widget = widgets.Dropdown(
    options=[
        "funny",
        "exciting",
        "relaxing",
        "emotional",
        "romantic",
        "thought_provoking",
        "suspenseful",
        "inspiring"
    ],
    value="funny",
    description="Experience:"
)

rating_widget = widgets.FloatSlider(
    value=6.0,
    min=0,
    max=10,
    step=0.5,
    description="Min Rating:"
)

display(
    user_id_widget,
    time_widget,
    language_widget,
    mood_widget,
    experience_widget,
    rating_widget
)

IntText(value=1, description='User ID:')

IntSlider(value=120, description='Time:', max=240, min=30, step=15)

Dropdown(description='Language:', options=(('English', 'en'), ('French', 'fr'), ('Japanese', 'ja'), ('Italian'…

Dropdown(description='Mood:', options=('happy', 'sad', 'relaxed', 'excited', 'romantic', 'thoughtful'), value=…

Dropdown(description='Experience:', options=('funny', 'exciting', 'relaxing', 'emotional', 'romantic', 'though…

FloatSlider(value=6.0, description='Min Rating:', max=10.0, step=0.5)

In [ ]:
# STEP 9.2 — Genre selection controls

genre_options = [
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

preferred_genres_widget = widgets.SelectMultiple(
    options=genre_options,
    value=("Comedy", "Adventure"),
    description="Prefer:",
    rows=8
)

avoided_genres_widget = widgets.SelectMultiple(
    options=genre_options,
    value=("Horror",),
    description="Avoid:",
    rows=8
)

print("Select one or more preferred genres:")
display(preferred_genres_widget)

print("\nSelect genres you want to avoid:")
display(avoided_genres_widget)

Select one or more preferred genres:


SelectMultiple(description='Prefer:', index=(4, 1), options=('Action', 'Adventure', 'Animation', 'Children', '…


Select genres you want to avoid:


SelectMultiple(description='Avoid:', index=(10,), options=('Action', 'Adventure', 'Animation', 'Children', 'Co…

In [ ]:
# STEP 9.3 — Recommendation button

recommend_button = widgets.Button(
    description="Recommend Movies",
    button_style="success",
    icon="film"
)

recommendation_output = widgets.Output()

def generate_recommendations(button):

    with recommendation_output:

        clear_output()

        preferred = list(preferred_genres_widget.value)
        avoided = list(avoided_genres_widget.value)

        print("🎬 CineSense AI")
        print("Generating personalized recommendations...\n")

        try:
            results = personalized_recommend_movies(
                user_id=user_id_widget.value,
                available_time=time_widget.value,
                language=language_widget.value,
                preferred_genres=preferred,
                avoided_genres=avoided,
                min_rating=rating_widget.value,
                mood=mood_widget.value,
                experience=experience_widget.value,
                top_n=10
            )

            if len(results) == 0:
                print("No movies matched these preferences.")
                print("Try increasing available time or reducing filters.")

            else:
                display(
                    results[
                        [
                            "clean_title",
                            "genres",
                            "runtime",
                            "vote_average",
                            "context_score",
                            "personalization_score",
                            "hybrid_score"
                        ]
                    ].reset_index(drop=True)
                )

        except Exception as e:
            print("Error:", e)


recommend_button.on_click(generate_recommendations)

display(recommend_button)
display(recommendation_output)

Button(button_style='success', description='Recommend Movies', icon='film', style=ButtonStyle())

Output()

In [ ]:
# STEP 9.4 — Explain recommendations

def explain_recommendation(row, preferred_genres):
    movie_genres = str(row["genres"]).split("|")

    matched = [
        genre for genre in preferred_genres
        if genre in movie_genres
    ]

    reasons = []

    if matched:
        reasons.append(
            "Matches your preferred genre(s): " + ", ".join(matched)
        )

    if row["vote_average"] >= 7.5:
        reasons.append("Highly rated movie")

    if row["context_score"] >= 8:
        reasons.append("Strong match for your current mood and experience")

    if row["personalization_score"] > 0:
        reasons.append("Matches your historical movie preferences")

    return " | ".join(reasons)


explained_results = final_user_recs.copy()

explained_results["why_recommended"] = explained_results.apply(
    lambda row: explain_recommendation(
        row,
        list(preferred_genres_widget.value)
    ),
    axis=1
)

display(
    explained_results[
        [
            "clean_title",
            "genres",
            "vote_average",
            "hybrid_score",
            "why_recommended"
        ]
    ]
)

,clean_title,genres,vote_average,hybrid_score,why_recommended
506,Aladdin,Adventure|Animation|Children|Comedy|Musical,7.655,0.847809,"Matches your preferred genre(s): Adventure, An..."
1177,Hercules,Adventure|Animation|Children|Comedy|Musical,7.496,0.838507,"Matches your preferred genre(s): Adventure, An..."
1390,Mulan,Adventure|Animation|Children|Comedy|Drama|Musi...,7.905,0.815805,"Matches your preferred genre(s): Adventure, An..."
578,Oliver & Company,Adventure|Animation|Children|Comedy|Musical,6.737,0.811403,"Matches your preferred genre(s): Adventure, An..."
5160,Shrek 2,Adventure|Animation|Children|Comedy|Musical|Ro...,7.326,0.776820,"Matches your preferred genre(s): Adventure, An..."
1545,"Little Mermaid, The",Animation|Children|Comedy|Musical|Romance,7.352,0.718788,"Matches your preferred genre(s): Animation, Co..."
4360,Finding Nemo,Adventure|Animation|Children|Comedy,7.820,0.708890,"Matches your preferred genre(s): Adventure, An..."
2144,Yellow Submarine,Adventure|Animation|Comedy|Fantasy|Musical,7.220,0.705934,"Matches your preferred genre(s): Adventure, An..."
8303,Frozen,Adventure|Animation|Comedy|Fantasy|Musical|Rom...,7.249,0.698021,"Matches your preferred genre(s): Adventure, An..."
6626,Enchanted,Adventure|Animation|Children|Comedy|Fantasy|Mu...,6.846,0.694745,"Matches your preferred genre(s): Adventure, An..."


In [ ]:
# STEP 9.5 — Recommendation Quality Evaluation

def evaluate_recommendations(
    recommendations,
    preferred_genres,
    avoided_genres
):
    total = len(recommendations)

    if total == 0:
        print("No recommendations available for evaluation.")
        return

    preferred_matches = 0
    avoided_matches = 0
    high_quality = 0

    for _, row in recommendations.iterrows():

        movie_genres = set(str(row["genres"]).split("|"))

        # Preferred genre match
        if movie_genres.intersection(preferred_genres):
            preferred_matches += 1

        # Avoided genre violation
        if movie_genres.intersection(avoided_genres):
            avoided_matches += 1

        # Highly rated recommendation
        if row["vote_average"] >= 7.0:
            high_quality += 1

    preference_match_rate = (preferred_matches / total) * 100
    avoidance_success_rate = ((total - avoided_matches) / total) * 100
    high_rating_rate = (high_quality / total) * 100

    print("CineSense AI — Recommendation Evaluation")
    print("----------------------------------------")
    print("Total recommendations:", total)

    print(
        f"Preferred Genre Match Rate: "
        f"{preference_match_rate:.2f}%"
    )

    print(
        f"Avoided Genre Success Rate: "
        f"{avoidance_success_rate:.2f}%"
    )

    print(
        f"High Rating Recommendation Rate: "
        f"{high_rating_rate:.2f}%"
    )

    overall_score = (
        preference_match_rate +
        avoidance_success_rate +
        high_rating_rate
    ) / 3

    print(f"\nOverall Recommendation Quality: {overall_score:.2f}%")

    return overall_score


evaluation_score = evaluate_recommendations(
    final_user_recs,
    set(preferred_genres_widget.value),
    set(avoided_genres_widget.value)
)

CineSense AI — Recommendation Evaluation
----------------------------------------
Total recommendations: 10
Preferred Genre Match Rate: 100.00%
Avoided Genre Success Rate: 100.00%
High Rating Recommendation Rate: 80.00%

Overall Recommendation Quality: 93.33%


In [ ]:
# STEP 9.6 — Test a Different User and Context

test_case_2 = personalized_recommend_movies(
    user_id=2,
    available_time=180,
    language="en",
    preferred_genres=["Action", "Thriller"],
    avoided_genres=["Romance"],
    min_rating=6.0,
    mood="exciting",
    experience="suspenseful",
    top_n=10
)

display(
    test_case_2[
        [
            "clean_title",
            "genres",
            "runtime",
            "vote_average",
            "context_score",
            "personalization_score",
            "final_score",
            "hybrid_score"
        ]
    ]
)

,clean_title,genres,runtime,vote_average,context_score,personalization_score,final_score,hybrid_score
7770,Sherlock Holmes: A Game of Shadows,Action|Adventure|Comedy|Crime|Mystery|Thriller,129.0,7.145,6.0,-0.078029,0.677035,0.773924
5850,Sin City,Action|Crime|Film-Noir|Mystery|Thriller,124.0,7.464,6.0,-0.175697,0.687377,0.717194
5167,Mindhunters,Action|Crime|Horror|Mystery|Thriller,106.0,6.457,8.0,-0.307156,0.803067,0.712075
3544,Mulholland Drive,Crime|Drama|Film-Noir|Mystery|Thriller,147.0,7.800,6.0,-0.216921,0.703644,0.701580
3873,Minority Report,Action|Crime|Mystery|Sci-Fi|Thriller,145.0,7.356,6.0,-0.199284,0.684111,0.699459
1813,Clue,Comedy|Crime|Mystery|Thriller,94.0,7.208,6.0,-0.196627,0.674910,0.694759
4940,Man on Fire,Action|Crime|Drama|Mystery|Thriller,146.0,7.471,6.0,-0.213805,0.688212,0.692818
951,Chinatown,Crime|Film-Noir|Mystery|Thriller,130.0,7.905,6.0,-0.223516,0.697180,0.692736
6016,Kiss Kiss Bang Bang,Comedy|Crime|Mystery|Thriller,103.0,7.161,6.0,-0.196627,0.671355,0.692270
1218,L.A. Confidential,Crime|Film-Noir|Mystery|Thriller,138.0,7.800,6.0,-0.223516,0.696233,0.692073


In [ ]:
# STEP 9.7 — Context Sensitivity Test

case_a = personalized_recommend_movies(
    user_id=1,
    available_time=120,
    language="en",
    preferred_genres=["Comedy", "Adventure"],
    avoided_genres=["Horror"],
    min_rating=6.0,
    mood="happy",
    experience="funny",
    top_n=10
)

case_b = personalized_recommend_movies(
    user_id=1,
    available_time=180,
    language="en",
    preferred_genres=["Action", "Thriller"],
    avoided_genres=["Romance"],
    min_rating=6.0,
    mood="exciting",
    experience="suspenseful",
    top_n=10
)

titles_a = set(case_a["clean_title"])
titles_b = set(case_b["clean_title"])

common_movies = titles_a.intersection(titles_b)

print("CineSense AI — Context Sensitivity Test")
print("----------------------------------------")
print("Context A: Happy + Funny + Comedy/Adventure")
print("Context B: Exciting + Suspenseful + Action/Thriller")

print("\nRecommendations in Context A:", len(case_a))
print("Recommendations in Context B:", len(case_b))
print("Common recommendations:", len(common_movies))

context_change_rate = (
    1 - len(common_movies) / min(len(case_a), len(case_b))
) * 100

print(
    f"Recommendation Change Rate: "
    f"{context_change_rate:.2f}%"
)

print("\nContext A Top Movies:")
print(case_a["clean_title"].tolist())

print("\nContext B Top Movies:")
print(case_b["clean_title"].tolist())

CineSense AI — Context Sensitivity Test
----------------------------------------
Context A: Happy + Funny + Comedy/Adventure
Context B: Exciting + Suspenseful + Action/Thriller

Recommendations in Context A: 10
Recommendations in Context B: 10
Common recommendations: 0
Recommendation Change Rate: 100.00%

Context A Top Movies:
['Aladdin', 'Hercules', 'Mulan', 'Oliver & Company', 'Shrek 2', 'Little Mermaid, The', 'Finding Nemo', 'Yellow Submarine', 'Frozen', 'Enchanted']

Context B Top Movies:
['Inception', 'Mulholland Drive', 'Godfather: Part III, The', 'Primal Fear', 'Man on Fire', 'Name of the Rose, The (Name der Rose, Der)', 'Twin Peaks: Fire Walk with Me', 'Fracture', 'Insomnia', 'Chinatown']


In [ ]:
# STEP 9.8 — Final System Evaluation Summary

print("=" * 60)
print("        CINESENSE AI — FINAL EVALUATION SUMMARY")
print("=" * 60)

print("\n1. RECOMMENDATION QUALITY")
print("-" * 40)
print("Preferred Genre Match Rate     : 100.00%")
print("Avoided Genre Success Rate     : 100.00%")
print("High Rating Recommendation Rate: 80.00%")
print("Overall Recommendation Quality : 93.33%")

print("\n2. CONTEXT SENSITIVITY")
print("-" * 40)
print("Context A Recommendations      :", len(case_a))
print("Context B Recommendations      :", len(case_b))
print("Common Recommendations         :", len(common_movies))
print(f"Recommendation Change Rate     : {context_change_rate:.2f}%")

print("\n3. SYSTEM CAPABILITIES")
print("-" * 40)

capabilities = [
    "User rating-history personalization",
    "Mood-aware recommendation",
    "Experience-aware recommendation",
    "Available-time filtering",
    "Language filtering",
    "Preferred genre matching",
    "Avoided genre filtering",
    "Minimum rating filtering",
    "TMDB popularity integration",
    "Hybrid recommendation scoring",
    "Previously watched movie removal",
    "Explainable recommendations"
]

for i, capability in enumerate(capabilities, 1):
    print(f"{i:02d}. {capability}")

print("\n" + "=" * 60)
print("CineSense AI recommendation pipeline successfully validated.")
print("=" * 60)

        CINESENSE AI — FINAL EVALUATION SUMMARY

1. RECOMMENDATION QUALITY
----------------------------------------
Preferred Genre Match Rate     : 100.00%
Avoided Genre Success Rate     : 100.00%
High Rating Recommendation Rate: 80.00%
Overall Recommendation Quality : 93.33%

2. CONTEXT SENSITIVITY
----------------------------------------
Context A Recommendations      : 10
Context B Recommendations      : 10
Common Recommendations         : 0
Recommendation Change Rate     : 100.00%

3. SYSTEM CAPABILITIES
----------------------------------------
01. User rating-history personalization
02. Mood-aware recommendation
03. Experience-aware recommendation
04. Available-time filtering
05. Language filtering
06. Preferred genre matching
07. Avoided genre filtering
08. Minimum rating filtering
09. TMDB popularity integration
10. Hybrid recommendation scoring
11. Previously watched movie removal
12. Explainable recommendations

CineSense AI recommendation pipeline successfully validated.
